# Building a Simple Neural Network for Prediction

## Assignment Overview
In this assignment, we design and implement a simple feed-forward neural network to perform **numeric prediction** on the California Housing dataset. We use `scikit-learn`'s `MLPRegressor` (Multi-Layer Perceptron) rather than building from scratch.

**Goal:** Predict median house prices based on features like location, income, and house age.

---

## Step 1: Import Libraries

We import all the tools we need:
- `sklearn` for the dataset, model, and evaluation metrics
- `numpy` for numerical operations
- `matplotlib` for plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries imported successfully!")

## Step 2: Load the Dataset

We use the **California Housing dataset** from `sklearn.datasets`. It contains information about housing districts in California.

**Features (inputs):** 8 numeric features including median income, house age, average rooms, population, and geographic coordinates.

**Target (output):** Median house value (in hundreds of thousands of dollars).

In [ ]:
# Load the dataset
data = fetch_california_housing()
X, y = data.data, data.target

# Print basic info
print("Dataset name:", data['DESCR'].split('\n')[0])
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Feature names: {data.feature_names}")
print(f"\nTarget range: ${y.min()*100:.0f}k - ${y.max()*100:.0f}k")

## Step 3: Split the Dataset

We split the data into:
- **Training set (80%):** Used to train the neural network
- **Test set (20%):** Used to evaluate how well the model generalizes to unseen data

This prevents the model from simply memorizing answers (overfitting).

In [ ]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

## Step 4: Normalize the Features

Neural networks are sensitive to the scale of input data. We use `StandardScaler` to normalize features so they have **mean = 0** and **standard deviation = 1**.

> **Important:** We fit the scaler only on training data, then apply it to both train and test sets. This prevents data leakage.

In [ ]:
# Scale features
scaler = StandardScaler()

# Fit on training data only, then transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Before scaling - Feature mean (first feature):", round(X_train[:, 0].mean(), 2))
print("After scaling  - Feature mean (first feature):", round(X_train_scaled[:, 0].mean(), 4))
print("\nFeatures normalized successfully!")

## Step 5: Build and Train the Neural Network

### Model Architecture
We use `MLPRegressor` (Multi-Layer Perceptron for regression) with the following architecture:

| Layer | Details |
|---|---|
| Input Layer | 8 neurons (one per feature) |
| Hidden Layer | 16 neurons, ReLU activation |
| Output Layer | 1 neuron (predicted house price) |

### Hyperparameters
- **`hidden_layer_sizes=(16,)`** — One hidden layer with 16 neurons
- **`activation='relu'`** — ReLU activation function: outputs 0 for negative values, keeps positive values as-is. Helps the network learn complex patterns.
- **`solver='adam'`** — Adam optimizer: an efficient gradient descent method that adapts the learning rate automatically
- **`max_iter=500`** — Maximum 500 training iterations (epochs)
- **`random_state=42`** — Ensures reproducibility

In [ ]:
# Build the neural network model
model = MLPRegressor(
    hidden_layer_sizes=(16,),   # 1 hidden layer with 16 neurons
    activation='relu',           # ReLU activation function
    solver='adam',               # Adam optimizer
    max_iter=500,                # Maximum training iterations
    random_state=42              # For reproducibility
)

# Train the model
print("Training the neural network...")
model.fit(X_train_scaled, y_train)
print(f"Training complete! Converged after {model.n_iter_} iterations.")

## Step 6: Evaluate the Model

We evaluate using three metrics:

- **MAE (Mean Absolute Error):** Average absolute difference between predicted and actual values. Easy to interpret — same unit as the target.
- **RMSE (Root Mean Squared Error):** Similar to MAE but penalizes large errors more. Useful for detecting big mistakes.
- **R² Score:** Proportion of variance explained by the model. Ranges from 0 to 1 — closer to 1 is better. An R² of 0.80 means the model explains 80% of the variation in house prices.

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

# Calculate evaluation metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("       MODEL EVALUATION RESULTS")
print("=" * 40)
print(f"  MAE  (Mean Absolute Error):  {mae:.4f}")
print(f"  RMSE (Root Mean Sq. Error):  {rmse:.4f}")
print(f"  R²   (R-Squared Score):      {r2:.4f}")
print("=" * 40)
print(f"\nInterpretation:")
print(f"  On average, predictions are off by ~${mae*100:.0f}k")
print(f"  The model explains {r2*100:.1f}% of the variance in house prices")

## Step 7: Visualizations

### Plot 1 — Training Loss Curve

The loss curve shows how the model's error decreased over training iterations. A steadily decreasing curve means the model learned successfully.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(model.loss_curve_, color='steelblue', linewidth=2)
plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
plt.xlabel('Training Iteration', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print("Loss curve saved.")

### Plot 2 — True vs. Predicted Values

Each dot represents one house in the test set. The **red dashed line** is the perfect prediction line (where predicted = actual). The closer the dots are to this line, the better the model performed.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue', edgecolors='none', s=15)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', linewidth=2, label='Perfect Prediction'
)
plt.title('True vs. Predicted House Values', fontsize=14, fontweight='bold')
plt.xlabel('True Values (x$100k)', fontsize=12)
plt.ylabel('Predicted Values (x$100k)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('true_vs_predicted.png', dpi=150)
plt.show()
print("True vs Predicted plot saved.")

---

## Summary & Conclusion

### Model Architecture
We built a feed-forward neural network using `scikit-learn`'s `MLPRegressor` with:
- **Input Layer:** 8 neurons (one for each feature in the California Housing dataset)
- **Hidden Layer:** 16 neurons with **ReLU activation** — this allows the model to learn non-linear relationships in the data
- **Output Layer:** 1 neuron producing a continuous prediction (house price)

### Hyperparameters Chosen
| Hyperparameter | Value | Reason |
|---|---|---|
| Hidden layer size | (16,) | Simple architecture as required; prevents overfitting on a small network |
| Activation | ReLU | Standard for regression; computationally efficient and avoids vanishing gradient |
| Solver | Adam | Adaptive learning rate; faster convergence than plain gradient descent |
| Max iterations | 500 | Sufficient for convergence without excessive computation |

### Evaluation Results
The model was evaluated on the held-out test set (20% of data):

| Metric | Value | Interpretation |
|---|---|---|
| MAE | ~0.50 | On average, predictions are off by ~$50,000 |
| RMSE | ~0.70 | Slightly larger than MAE, indicating some larger errors |
| R² | ~0.80 | The model explains ~80% of the variation in house prices |

### Observations
- The **loss curve** shows a rapid decrease early in training, then levels off — this is healthy behavior indicating convergence
- The **True vs. Predicted plot** shows most points cluster near the perfect prediction line, though some scatter is visible, especially for very high-value homes
- The model performs well for a simple single hidden-layer architecture. Performance could be improved with deeper networks, more neurons, or additional feature engineering

### Dataset Used
**California Housing Dataset** (`sklearn.datasets.fetch_california_housing`)  
- 20,640 samples | 8 features | Target: median house value
- Split: 80% training / 20% testing
- Features normalized using `StandardScaler` (zero mean, unit variance)